<a href="https://colab.research.google.com/github/Aggarwalmansi/GENAI/blob/main/Copy_of_RNN_to_Transformers_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers Example

In [1]:
!pip install -q transformers datasets accelerate

from transformers import AutoTokenizer , AutoModelForSequenceClassification , Trainer , TrainingArguments
from datasets import Dataset

In [2]:
data = {
    "text": [
        "I am so happy today",
        "This is the best day",
        "I feel terrible and sad",
        "This is a disaster",
        "I am feeling great",       # Added 'great'
        "What a wonderful morning", # Added 'wonderful'
        "I hate this movie",        # Added 'hate'
        "This is awful",            # Added 'awful'
        "I love this place",        # Added 'love'
        "I am depressed"            # Added 'depressed'
    ],
    "label": [1, 1, 0, 0, 1, 1, 0, 0, 1, 0]
}
dataset = Dataset.from_dict(data)

In [3]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=16)

In [5]:
tokenizer_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [6]:
args = TrainingArguments(
    output_dir="simple_model_v2",
    num_train_epochs=15,
    per_device_train_batch_size=4,
    report_to="none"
)

In [7]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenizer_dataset
)

In [8]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=45, training_loss=0.20641350216335722, metrics={'train_runtime': 7.6552, 'train_samples_per_second': 19.595, 'train_steps_per_second': 5.878, 'total_flos': 620940931200.0, 'train_loss': 0.20641350216335722, 'epoch': 15.0})

In [9]:
test_sentences = ["I am feeling great!" , "This is the worst day ever."]
for text in test_sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits
    prediction = logits.argmax().item()
    label = "Happy" if prediction == 1 else "Sad"
    print(f"Input: '{text}' -> Prediction: {label}")

Input: 'I am feeling great!' -> Prediction: Happy
Input: 'This is the worst day ever.' -> Prediction: Happy
